## 1. Import required libraries

In [6]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import random
import matplotlib.pyplot as plt

In [3]:
torch.__version__

'2.7.0+cu128'

## 2. Setup Device-Agnostic Code

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [9]:
torch.cuda.is_available()

True

In [22]:
!nvidia-smi

Sun Mar 29 20:30:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.59                 Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   69C    P0            108W /  135W |     889MiB /   8188MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
print(f"Using device: {device}")

Using device: cuda


## 3. Set seed

In [27]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

## 4. Setting the hyperparameters

In [ ]:
BATCH_SIZE  = 128
EPOCHS = 10
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_SIZE = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1

## 5. Define Image transformations 

In [30]:
transform = transforms.Compose({
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    # 1. Helps the model to converge faster and improve the performance
    # 2. Helps to make the numerical computations stable
    
})

## 6. Getting the dataset

In [ ]:
train_dataset = datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)

In [ ]:
train_dataset

In [ ]:
test_dataset = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

In [ ]:
test_dataset

## 7. Converting datasets to Dataloaders
Right now, our data is in the form of PyTorch Datasets.

Dataloader turns our data into batches or mini-batches

Why do we do this?

1. It is more computationally efficient, as in, our computing hardware may not be able to look (store in memory) at 50k images in one hit. So we break it into 128 images at a time. (batch size of 128)
2. It gives our Neural network more chances to update its gradients per epoch

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
print(f"DataLoader: {train_dataloader}, {test_dataloader}")
print(f"Length of train dataloader: {len(train_dataloader)} batches of size {BATCH_SIZE}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of size {BATCH_SIZE}")

## 8. Building Vision Transformer Model from Scratch

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=4, embed_dim=256, img_size=32):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels=in_channels, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim)) # 1 for cls token, so it acts just like an additional patch. If total 64 patches (each patch has 4 image patches), then we will have 65 tokens (64 for patches and 1 for cls token)
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + self.num_patches, embed_dim)) # 1 for cls token